# Task 3 (Medium): Content-Based Recommendation System

## 1. Introduction

This section focuses on developing a content-based recommendation system using the Netflix titles dataset. The objective is to identify and recommend movies and TV shows with similar characteristics based on their available content attributes, including genres, descriptions, cast, and directors.

Unlike collaborative filtering, which relies on user interactions and preferences, a content-based approach uses information about the items themselves to estimate similarity. This makes it suitable for exploring recommendations when user viewing histories and ratings are unavailable.

The workflow applies Natural Language Processing (NLP), feature engineering, and similarity analysis to transform Netflix title information into a recommendation model.

## 2. Objectives

The main objectives of this task are to:

* Identify and extract relevant features from the Netflix dataset.
* Prepare and preprocess textual data for analysis.
* Convert text features into numerical representations using TF-IDF.
* Calculate content similarity using cosine similarity.
* Generate ranked recommendations for selected Netflix titles.
* Evaluate the relevance, consistency, and diversity of the recommendations.
* Identify the limitations of the model and opportunities for improvement.

## 3. Task Workflow

### Step 1: Extract Relevant Content Features

Inspect the available dataset columns and select attributes suitable for representing each title, including title, content type, genres, description, cast, and director where available. Examine missing values and determine which features should be included in the model.

### Step 2: Perform Text Preprocessing

Prepare the selected features by handling missing values, standardizing text, and combining relevant attributes into a consistent content representation for each title.

### Step 3: Convert Text into Numerical Features

Apply Term Frequency–Inverse Document Frequency (TF-IDF) vectorization to transform the prepared textual content into numerical feature vectors that can be compared mathematically.

### Step 4: Calculate Content Similarity

Use cosine similarity to measure the similarity between title representations and rank potential recommendations according to their similarity to a selected title.

### Step 5: Generate Recommendations

Develop a reusable recommendation function that accepts a Netflix title and returns a ranked list of similar movies or TV shows. Test the function using selected titles and handle missing or ambiguous title matches appropriately.

### Step 6: Evaluate Recommendation Quality

Review the generated recommendations to assess content relevance, genre consistency, similarity rankings, and diversity. Discuss the limitations of evaluating recommendations without user–item interaction data.

### Step 7: Summarize Findings and Limitations

Document the model's performance, useful recommendation patterns, technical limitations, and possible improvements to the recommendation approach.

## 4. Tools and Techniques

The following tools and techniques will be used:

* **Python:** Data manipulation and implementation.
* **Pandas:** Feature selection, preprocessing, and data management.
* **Scikit-learn:** TF-IDF vectorization and cosine similarity.
* **Natural Language Processing (NLP):** Text preparation and representation.
* **Content-Based Filtering:** Recommendation generation using item attributes.
* **Exploratory Evaluation:** Inspection of recommendation relevance and diversity.

## 5. Expected Outcome

The expected outcome is a functional content-based recommendation system that uses the available Netflix title attributes to identify and rank similar content. The resulting model will demonstrate how textual feature engineering and similarity analysis can support content discovery.

The model will be evaluated primarily on content similarity and qualitative relevance. Since the dataset does not contain user–item interaction histories, the results will not be interpreted as direct evidence of personalized user preferences or real-world recommendation effectiveness.


In [1]:
# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Calculate similarity between title representations
from sklearn.metrics.pairwise import cosine_similarity

# Optional evaluation and text-feature utilities
from sklearn.preprocessing import normalize

In [2]:
# importing dataset
df = pd.read_csv('netflix_cleaned.csv')
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,date_added_year,date_added_month,date_added_day,duration_value,duration_unit,genre_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,25,90,min,1
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,24,1,Season,3
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,24,1,Season,3
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,22,91,min,2
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,24,125,min,3


In [6]:
# Check missing values in each column
display(
    df.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing Count")
)


,Missing Count
director,2588
country,287
type,0
show_id,0
title,0
date_added,0
release_year,0
rating,0
duration,0
listed_in,0


### Creating a feature selection dataframe

In [7]:
# Select the columns that may help represent each title
feature_columns = [
    "title",
    "type",
    "listed_in",
    "director",
    "country"
]

# Create a separate DataFrame so the original df remains unchanged
recommendation_df = df[feature_columns].copy()

# Check the selected data
display(recommendation_df.head())

# Confirm the number of rows and columns
print("Recommendation dataset shape:", recommendation_df.shape)

# Check missing values in the selected features
display(
    recommendation_df.isnull()
    .sum()
    .to_frame(name="Missing Count")
)

,title,type,listed_in,director,country
0,Dick Johnson Is Dead,Movie,Documentaries,Kirsten Johnson,United States
1,Ganglands,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq,France
2,Midnight Mass,TV Show,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan,United States
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies",Bruno Garotti,Brazil
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies",Haile Gerima,United States


Recommendation dataset shape: (8790, 5)


,Missing Count
title,0
type,0
listed_in,0
director,2588
country,287


#### Step 2: Text Preprocessing

Text preprocessing prepares the selected content attributes for similarity analysis. The `listed_in` column contains genre and category information, while `director` and `country` provide additional information that may help distinguish titles.

Missing values in optional features will be handled without removing titles from the dataset. Text will be standardized to lowercase and stripped of unnecessary whitespace to improve consistency before the features are combined into a representation suitable for TF-IDF vectorization.

The original `df` and `recommendation_df` will be preserved, with preprocessing performed on a separate working DataFrame.


In [8]:
# Create a separate working copy for preprocessing
processed_df = recommendation_df.copy()

# Fill missing values in optional text features
processed_df["director"] = processed_df["director"].fillna("")
processed_df["country"] = processed_df["country"].fillna("")

# Standardize text by converting it to lowercase and trimming whitespace
text_columns = ["type", "listed_in", "director", "country"]

for column in text_columns:
    processed_df[column] = (
        processed_df[column]
        .astype(str)
        .str.lower()
        .str.strip()
    )

# Check that missing values have been handled
display(
    processed_df[text_columns]
    .isnull()
    .sum()
    .to_frame(name="Missing Count")
)

# Preview the preprocessed features
display(processed_df.head())

,Missing Count
type,0
listed_in,0
director,0
country,0


,title,type,listed_in,director,country
0,Dick Johnson Is Dead,movie,documentaries,kirsten johnson,united states
1,Ganglands,tv show,"crime tv shows, international tv shows, tv act...",julien leclercq,france
2,Midnight Mass,tv show,"tv dramas, tv horror, tv mysteries",mike flanagan,united states
3,Confessions of an Invisible Girl,movie,"children & family movies, comedies",bruno garotti,brazil
4,Sankofa,movie,"dramas, independent movies, international movies",haile gerima,united states


### Step 3: Combine Content Features

The cleaned content attributes will be combined into a single text field for each Netflix title. This creates a unified representation of each item using its content type, genres and categories, director, and country.

Combining these attributes allows the TF-IDF vectorizer to represent titles using multiple characteristics rather than relying on a single feature. The resulting text field will serve as the input for numerical feature extraction and cosine similarity calculations.


In [9]:
# Combine the selected content attributes into one text field per title
processed_df["content_features"] = (
    processed_df["type"] + " " +
    processed_df["listed_in"] + " " +
    processed_df["director"] + " " +
    processed_df["country"]
)

# Inspect the combined text representation
display(
    processed_df[["title", "content_features"]].head()
)

,title,content_features
0,Dick Johnson Is Dead,movie documentaries kirsten johnson united states
1,Ganglands,"tv show crime tv shows, international tv shows..."
2,Midnight Mass,"tv show tv dramas, tv horror, tv mysteries mik..."
3,Confessions of an Invisible Girl,"movie children & family movies, comedies bruno..."
4,Sankofa,"movie dramas, independent movies, internationa..."


### Step 4: TF-IDF Vectorization

TF-IDF converts the combined text features into numerical vectors, assigning weights to words based on their importance across the dataset.

These vectors will enable us to compare Netflix titles using cosine similarity and identify content with similar characteristics.


In [10]:
# Initialize TF-IDF to convert title content into numerical features
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Learn the vocabulary and calculate TF-IDF weights
tfidf_matrix = tfidf.fit_transform(
    processed_df["content_features"]
)

# Inspect the dimensions of the resulting matrix
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of titles:", tfidf_matrix.shape[0])
print("Number of text features:", tfidf_matrix.shape[1])

TF-IDF matrix shape: (8790, 5000)
Number of titles: 8790
Number of text features: 5000


#### Calculate Cosine Similarity

Cosine similarity measures how similar two titles are based on their TF-IDF vectors. Scores closer to 1 indicate greater similarity, while scores closer to 0 indicate less similarity.

In [11]:
# Calculate pairwise similarity scores between all Netflix titles
cosine_sim = cosine_similarity(tfidf_matrix)

# Confirm the dimensions of the similarity matrix
print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (8790, 8790)


### Generate Recommendations

Create a reusable function that takes a Netflix title and returns the most similar titles, ranked by cosine similarity. The selected title itself will be excluded from the results.


In [12]:
# Create a lookup from lowercase titles to their row indices
title_indices = pd.Series(
    processed_df.index,
    index=processed_df["title"].str.lower()
).drop_duplicates()


def recommend_titles(title, n=10):
    """Return the top n titles most similar to the selected Netflix title."""

    # Find the selected title, ignoring capitalization
    title_key = title.strip().lower()

    if title_key not in title_indices.index:
        return f"Title '{title}' was not found in the dataset."

    # Get the dataset index of the selected title
    idx = title_indices[title_key]

    # Pair each title index with its cosine similarity score
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Rank titles from most to least similar, excluding the selected title
    similarity_scores = sorted(
        similarity_scores,
        key=lambda item: item[1],
        reverse=True
    )

    similarity_scores = [
        item for item in similarity_scores if item[0] != idx
    ][:n]

    # Retrieve recommended titles and their similarity scores
    recommendations = processed_df.iloc[
        [item[0] for item in similarity_scores]
    ][["title", "type", "listed_in"]].copy()

    recommendations["similarity_score"] = [
        round(item[1], 3) for item in similarity_scores
    ]

    return recommendations.reset_index(drop=True)

In [13]:
# Generate 10 recommendations for a sample Netflix title
recommend_titles("Midnight Mass")

,title,type,listed_in,similarity_score
0,Brand New Cherry Flavor,tv show,"tv dramas, tv horror, tv mysteries",0.740
1,The Haunting of Bly Manor,tv show,"tv dramas, tv horror, tv mysteries",0.740
2,Ratched,tv show,"tv dramas, tv horror, tv mysteries",0.740
3,The Haunting of Hill House,tv show,"tv dramas, tv horror, tv mysteries",0.740
4,The Originals,tv show,"tv dramas, tv horror, tv mysteries",0.740
5,Before I Wake,movie,"horror movies, thrillers",0.695
6,Gerald's Game,movie,"horror movies, thrillers",0.695
7,Hush,movie,"horror movies, thrillers",0.695
8,American Horror Story,tv show,"tv horror, tv mysteries, tv thrillers",0.695
9,Hannibal,tv show,"tv horror, tv mysteries, tv thrillers",0.695


**One important observation**

The first five recommendations have identical similarity scores. This makes sense because several titles share the same genres, and our model uses only a limited set of metadata features. The director and country may help distinguish titles, but the shared genre features appear to dominate these results.

What I have learned is that titles can receive identical scores when their available metadata overlaps heavily.

In [14]:
# Test recommendations using a different Netflix title
recommend_titles("The Conjuring")

,title,type,listed_in,similarity_score
0,Insidious,movie,"horror movies, thrillers",1.000
1,The Conjuring 2,movie,horror movies,0.870
2,The Bye Bye Man,movie,"horror movies, thrillers",0.515
3,Final Destination,movie,"cult movies, horror movies, thrillers",0.501
4,Secrets of the Saqqara Tomb,movie,documentaries,0.500
5,Final Destination 3,movie,horror movies,0.481
6,Power Rangers: Megaforce: The Robo Knight Befo...,movie,children & family movies,0.463
7,Guy Martin: Last Flight of the Vulcan Bomber,movie,"documentaries, international movies",0.435
8,Straight Up,movie,"comedies, independent movies, lgbtq movies",0.415
9,Perfect Stranger,movie,thrillers,0.410


#### **What the results tell us**

**1. The model identifies relevant horror titles.**

Insidious scores 1.000, meaning its TF-IDF vector is identical to The Conjuring's under our selected features.

The Conjuring 2 scores 0.870, another relevant recommendation.

The Bye Bye Man and the Final Destination films also share horror-related categories.

A score of 1.000 does not necessarily mean two titles are identical in plot or overall content. It means their numerical feature representations point in exactly the same direction.

**2. Some recommendations are clearly less relevant.**

Secrets of the Saqqara Tomb and Guy Martin: Last Flight of the Vulcan Bomber are documentaries, while The Conjuring is a horror movie. This suggests that shared general features, such as the word movie, can contribute to similarity even when genres differ.

**3. Our model needs refinement.**

Because our dataset doesn't contain descriptions or cast information, the model relies heavily on genres, directors, and countries. We should assess how much each feature contributes before deciding how to improve it.

### Inspect Recommendation Quality

Examine the content features of selected recommendations to understand why the model considers them similar. This helps identify relevant matches and weaknesses caused by limited or shared metadata.

In [15]:
# Inspect the source features of the selected title and its recommendations
titles_to_inspect = [
    "The Conjuring",
    "Insidious",
    "The Conjuring 2",
    "Secrets of the Saqqara Tomb",
    "Guy Martin: Last Flight of the Vulcan Bomber"
]

display(
    processed_df[
        processed_df["title"].isin(titles_to_inspect)
    ][["title", "type", "listed_in", "director", "country", "content_features"]]
)

,title,type,listed_in,director,country,content_features
1100,Insidious,movie,"horror movies, thrillers",james wan,united states,"movie horror movies, thrillers james wan unite..."
1218,The Conjuring,movie,"horror movies, thrillers",james wan,united states,"movie horror movies, thrillers james wan unite..."
1219,The Conjuring 2,movie,horror movies,james wan,canada,movie horror movies james wan canada
1544,Secrets of the Saqqara Tomb,movie,documentaries,james tovell,united states,movie documentaries james tovell united states
5010,Guy Martin: Last Flight of the Vulcan Bomber,movie,"documentaries, international movies",james woodroffe,united kingdom,"movie documentaries, international movies jame..."


#### Finding: Recommendation Quality and Feature Limitations

The initial recommendations showed strong genre alignment for some titles but also included less relevant matches. Inspection revealed that titles could receive high similarity scores by sharing metadata such as content type, director, or country, even when their genres differed. For example, *Insidious* had identical selected features to *The Conjuring*, resulting in a cosine similarity score of 1.000.

This highlights a limitation of relying on combined metadata alone. To improve relevance, the recommendation function was refined to prioritize genre overlap and restrict recommendations to the same content type. These changes aim to produce more consistent recommendations, although further evaluation is needed to determine their effectiveness.


### Refining the Recommendation Filtering

Refine the recommendation function by restricting results to the same content type and prioritizing titles with overlapping genres. This aims to improve relevance while retaining cosine similarity as the ranking measure.

In [16]:
def recommend_titles(title, n=10):
    """Recommend titles of the same type, prioritizing genre overlap."""

    # Find the selected title, ignoring capitalization
    title_key = title.strip().lower()

    if title_key not in title_indices.index:
        return f"Title '{title}' was not found in the dataset."

    # Get the source title and its dataset index
    idx = title_indices[title_key]
    source = processed_df.loc[idx]

    # Identify the source title's genres
    source_genres = set(source["listed_in"].split(", "))

    # Calculate similarity scores against all titles
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Keep only other titles of the same type
    candidates = []
    for candidate_idx, score in similarity_scores:
        if candidate_idx == idx:
            continue

        candidate = processed_df.iloc[candidate_idx]

        if candidate["type"] != source["type"]:
            continue

        candidate_genres = set(candidate["listed_in"].split(", "))
        genre_overlap = len(source_genres & candidate_genres)

        # Keep titles with at least one shared genre
        if genre_overlap > 0:
            candidates.append(
                (candidate_idx, score, genre_overlap)
            )

    # Rank by genre overlap first, then cosine similarity
    candidates.sort(
        key=lambda item: (item[2], item[1]),
        reverse=True
    )

    # Return the top recommendations with their scores
    top_candidates = candidates[:n]

    recommendations = processed_df.iloc[
        [item[0] for item in top_candidates]
    ][["title", "type", "listed_in"]].copy()

    recommendations["similarity_score"] = [
        round(item[1], 3) for item in top_candidates
    ]
    recommendations["shared_genres"] = [
        item[2] for item in top_candidates
    ]

    return recommendations.reset_index(drop=True)

In [17]:
# Recheck recommendations for The Conjuring
recommend_titles("The Conjuring")

,title,type,listed_in,similarity_score,shared_genres
0,Insidious,movie,"horror movies, thrillers",1.000,2
1,The Bye Bye Man,movie,"horror movies, thrillers",0.515,2
2,Final Destination,movie,"cult movies, horror movies, thrillers",0.501,2
3,Under the Shadow,movie,"horror movies, international movies, thrillers",0.408,2
4,Emelie,movie,"horror movies, thrillers",0.397,2
5,Kanika,movie,"horror movies, international movies, thrillers",0.397,2
6,Death Note,movie,"horror movies, thrillers",0.361,2
7,Case 39,movie,"horror movies, thrillers",0.340,2
8,Jaws 3,movie,"action & adventure, horror movies, thrillers",0.323,2
9,13 Cameras,movie,"horror movies, independent movies, thrillers",0.312,2


#### Finding: Improved Recommendation Relevance

After refining the recommendation function, all ten recommendations for *The Conjuring* were movies sharing both the horror and thriller genres with the source title. *Insidious* ranked first with a cosine similarity score of 1.000, followed by *The Bye Bye Man* at 0.515.

Restricting recommendations to the same content type and prioritizing genre overlap reduced the less relevant cross-genre matches observed in the initial results. However, shared genres do not guarantee similar plots or viewer preferences, so further qualitative evaluation remains necessary.


In [18]:
# Regenerate 10 recommendations for a sample Netflix title
recommend_titles("Midnight Mass")

,title,type,listed_in,similarity_score,shared_genres
0,Brand New Cherry Flavor,tv show,"tv dramas, tv horror, tv mysteries",0.740,3
1,The Haunting of Bly Manor,tv show,"tv dramas, tv horror, tv mysteries",0.740,3
2,Ratched,tv show,"tv dramas, tv horror, tv mysteries",0.740,3
3,The Haunting of Hill House,tv show,"tv dramas, tv horror, tv mysteries",0.740,3
4,The Originals,tv show,"tv dramas, tv horror, tv mysteries",0.740,3
5,The Mist,tv show,"tv dramas, tv horror, tv mysteries",0.669,3
6,Lost Girl,tv show,"tv dramas, tv horror, tv mysteries",0.669,3
7,Penny Dreadful,tv show,"tv dramas, tv horror, tv mysteries",0.618,3
8,American Horror Story,tv show,"tv horror, tv mysteries, tv thrillers",0.695,2
9,Hannibal,tv show,"tv horror, tv mysteries, tv thrillers",0.695,2


#### Finding: Consistency Across Content Types

Testing the refined function with *Midnight Mass* produced ten TV show recommendations. The first eight shared all three of its genres: TV dramas, TV horror, and TV mysteries. The remaining two shared two genres, demonstrating that the function can rank titles with different levels of genre overlap.

Together with the *The Conjuring* test, these results suggest that filtering by content type and prioritizing shared genres improves the consistency of recommendations. However, identical metadata can still produce tied similarity scores, and genre overlap alone cannot establish actual viewer satisfaction.


In [19]:
# Split genre categories and count how often each appears
genre_counts = (
    processed_df["listed_in"]
    .str.split(", ")
    .explode()
    .value_counts()
)

# Display the 15 most common genres
display(genre_counts.head(15).to_frame("title_count"))

,title_count
listed_in,
international movies,2752
dramas,2426
comedies,1674
international tv shows,1349
documentaries,869
action & adventure,859
tv dramas,762
independent movies,756
children & family movies,641


In [20]:
# Select genres that cover different content categories
genres_to_test = [
    "comedies",
    "documentaries",
    "action & adventure",
    "romantic movies",
    "crime tv shows",
    "kids' tv",
    "tv horror"
]

for genre in genres_to_test:
    # Find titles whose genre categories contain the selected genre
    matches = processed_df[
        processed_df["listed_in"].str.split(", ").apply(
            lambda categories: genre in categories
        )
    ]

    # Display up to three examples for each genre
    print(f"\n{genre.title()} — {len(matches)} titles")
    display(matches[["title", "type", "listed_in"]].head(3))


Comedies — 1674 titles


,title,type,listed_in
3,Confessions of an Invisible Girl,movie,"children & family movies, comedies"
6,The Starling,movie,"comedies, dramas"
7,Motu Patlu in the Game of Zones,movie,"children & family movies, comedies, music & mu..."



Documentaries — 869 titles


,title,type,listed_in
0,Dick Johnson Is Dead,movie,documentaries
25,Europe's Most Dangerous Man: Otto Skorzeny in ...,movie,"documentaries, international movies"
66,Dolly Parton: A MusiCares Tribute,movie,"documentaries, music & musicals"



Action & Adventure — 859 titles


...

,title,type,listed_in
1,Ganglands,tv show,"crime tv shows, international tv shows, tv act..."
18,Crime Stories: India Detectives,tv show,"british tv shows, crime tv shows, docuseries"
24,Bangkok Breaking,tv show,"crime tv shows, international tv shows, tv act..."



Kids' Tv — 448 titles


,title,type,listed_in
20,True: Magical Friends,tv show,kids' tv
21,True: Wonderful Wishes,tv show,kids' tv
22,Dance & Sing with True,tv show,kids' tv



Tv Horror — 75 titles


,title,type,listed_in
2,Midnight Mass,tv show,"tv dramas, tv horror, tv mysteries"
456,Goodnight DJ 1,tv show,"international tv shows, tv dramas, tv horror"
830,Trese,tv show,"anime series, crime tv shows, tv horror"


In [21]:
# Testing more titles to inspect
recommend_titles("Afterlife of the party")

,title,type,listed_in,similarity_score,shared_genres
0,Big Stone Gap,movie,"comedies, romantic movies",0.419,2
1,Jumping the Broom,movie,"comedies, romantic movies",0.419,2
2,Santa Girl,movie,"children & family movies, comedies, romantic m...",0.307,2
3,Getting Played,movie,"comedies, romantic movies",0.296,2
4,Exit Strategy,movie,"comedies, romantic movies",0.295,2
5,Holidate,movie,"comedies, romantic movies",0.287,2
6,Maniyarayile Ashokan,movie,"comedies, international movies, romantic movies",0.286,2
7,Mumbai Matinee,movie,"comedies, international movies, romantic movies",0.286,2
8,Chadi Jawani Budhe Nu,movie,"comedies, international movies, romantic movies",0.286,2
9,The Good Catholic,movie,"comedies, dramas, romantic movies",0.273,2


#### Finding: Comedy and Romance Recommendations

When *Afterlife of the Party* was used as the source title, the refined recommendation function returned ten movies sharing both comedy and romance categories. *Big Stone Gap* and *Jumping the Broom* ranked highest, each with a cosine similarity score of 0.419. These results demonstrate consistent genre-based matching, although shared categories and tied scores do not necessarily indicate similar plots or audience appeal.


In [22]:
recommend_titles("Dolly Parton: A MusiCares Tribute")

,title,type,listed_in,similarity_score,shared_genres
0,Ma Rainey's Black Bottom: A Legacy Brought to ...,movie,"documentaries, music & musicals",1.000,2
1,John Mellencamp: Plain Spoken,movie,"documentaries, music & musicals",1.000,2
2,Zulu Man in Japan,movie,"documentaries, international movies, music & m...",0.957,2
3,Wave of Cinema: Surat dari Timur,movie,"documentaries, international movies, music & m...",0.813,2
4,Dark City Beneath the Beat,movie,"documentaries, music & musicals",0.803,2
5,Once in a Lifetime Sessions with George Ezra,movie,"documentaries, music & musicals",0.803,2
6,Once in a Lifetime Sessions with OneRepublic,movie,"documentaries, music & musicals",0.803,2
7,LA Originals,movie,"documentaries, international movies, music & m...",0.763,2
8,One Heart: The A.R. Rahman Concert Film,movie,"documentaries, international movies, music & m...",0.748,2
9,The Show Must Go On: The Queen + Adam Lambert ...,movie,"documentaries, music & musicals",0.743,2


#### Finding: Documentary and Music Recommendations

Using *Dolly Parton: A MusiCares Tribute* as the input returned ten movie recommendations that shared both the **documentaries** and **music & musicals** categories. *Ma Rainey's Black Bottom: A Legacy Brought to You by Netflix* and *John Mellencamp: Plain Spoken* each received a similarity score of **1.000**, indicating identical TF-IDF feature representations based on the selected metadata.

The results demonstrate consistent genre-based matching for music documentaries. However, identical scores do not necessarily mean two titles cover the same subject or appeal to the same audience. This highlights a limitation of the current model: recommendations depend on available metadata rather than detailed plot descriptions or themes.


In [23]:
recommend_titles("The Stronghold")

,title,type,listed_in,similarity_score,shared_genres
0,Abdo Mota,movie,"action & adventure, dramas, international movies",0.285,3
1,My True Friend,movie,"action & adventure, dramas, international movies",0.270,3
2,The Great Battle,movie,"action & adventure, dramas, international movies",0.254,3
3,Mersal,movie,"action & adventure, dramas, international movies",0.233,3
4,Mahabharat,movie,"action & adventure, dramas, international movies",0.231,3
5,Baahubali: The Beginning (English Version),movie,"action & adventure, dramas, international movies",0.224,3
6,Baahubali 2: The Conclusion (Malayalam Version),movie,"action & adventure, dramas, international movies",0.224,3
7,Baahubali 2: The Conclusion (Tamil Version),movie,"action & adventure, dramas, international movies",0.224,3
8,Baahubali: The Beginning (Malayalam Version),movie,"action & adventure, dramas, international movies",0.224,3
9,Baahubali: The Beginning (Tamil Version),movie,"action & adventure, dramas, international movies",0.224,3


#### Finding: Action and Adventure Recommendations

Using *The Stronghold* as the input returned ten movie recommendations sharing the same three categories: action & adventure, dramas, and international movies. *Abdo Mota* ranked first with a similarity score of 0.285, followed by *My True Friend* at 0.270 and *The Great Battle* at 0.254.

The results demonstrate consistent genre matching across action-oriented international films. However, the relatively low similarity scores compared with some earlier tests suggest that the titles share fewer or less distinctive metadata features. Several *Baahubali* entries also appeared in different language versions, highlighting that the recommender may return multiple versions of related content rather than a diverse selection of titles.


In [24]:
recommend_titles("True: Wonderful Wishes")

,title,type,listed_in,similarity_score,shared_genres
0,True: Magical Friends,tv show,kids' tv,1.000,1
1,Dance & Sing with True,tv show,kids' tv,1.000,1
2,True and the Rainbow Kingdom,tv show,kids' tv,1.000,1
3,True: Terrific Tales,tv show,kids' tv,0.945,1
4,Power Rangers Dino Fury,tv show,kids' tv,0.419,1
5,Sharkdog,tv show,kids' tv,0.419,1
6,Go! Go! Cory Carson,tv show,kids' tv,0.419,1
7,Mother Goose Club,tv show,kids' tv,0.419,1
8,Fast & Furious Spy Racers,tv show,kids' tv,0.419,1
9,Gabby's Dollhouse,tv show,kids' tv,0.419,1


#### Finding: Children's Content Recommendations

Using *True: Wonderful Wishes* as the input returned ten TV show recommendations categorized as kids' TV. *True: Magical Friends*, *Dance & Sing with True*, and *True and the Rainbow Kingdom* each received a similarity score of 1.000, while *True: Terrific Tales* scored 0.945.

These high scores suggest that the titles share identical or very similar representations across the selected metadata features. However, the remaining recommendations received a score of 0.419 despite sharing the same category. This illustrates how limited genre information can group children's shows together without establishing that they have similar themes, storylines, or target audiences.


### Overall Recommendation System Evaluation

The content-based recommendation system was evaluated using six titles spanning horror, television dramas, comedy-romance, music documentaries, action and adventure, and children's content.

The tests showed that the system consistently returned titles sharing the source title's content type and genre categories. Prioritizing genre overlap also reduced recommendations from unrelated content categories.

However, the evaluation revealed several limitations. Titles with identical metadata representations could receive a perfect similarity score, multiple language versions of the same film could appear together, and broad genre categories sometimes produced recommendations with limited thematic similarity. These findings indicate that the system is useful for metadata-based discovery but would benefit from richer content features and more systematic evaluation.


## Limitations and Future Improvements

### 1. Limited Content Features

The recommendation system uses content type, genres, director, and country to represent each title. These features provide useful metadata but do not capture plot details, themes, cast, or audience preferences.

**Potential improvement:** Incorporate plot descriptions and cast information where reliable data is available to build richer content representations.

### 2. Identical Similarity Scores

Some titles received a cosine similarity score of 1.000 because their selected metadata produced identical TF-IDF representations. This does not necessarily mean the titles have identical storylines or appeal to the same audience.

**Potential improvement:** Introduce more discriminative features and inspect duplicate or near-duplicate metadata representations.

### 3. Limited Recommendation Diversity

The system sometimes returned multiple language versions of the same film or several closely related titles. Although these recommendations may be relevant, they can reduce variety in the results.

**Potential improvement:** Apply a diversity-aware ranking strategy or filter near-duplicate titles when appropriate.

### 4. Broad Genre-Based Matching

Titles sharing broad categories may be recommended together even when their stories or themes differ. Genre overlap improves consistency but does not guarantee semantic relevance.

**Potential improvement:** Include richer textual features and evaluate recommendations against human relevance judgments.

### 5. Limited Evaluation

The current evaluation uses a small set of manually selected titles. It demonstrates how the system behaves in representative examples but does not establish its overall recommendation quality.

**Potential improvement:** Expand testing across more titles and content categories, then assess relevance, diversity, and ranking quality using a consistent evaluation framework.

## Future Development

Future versions could incorporate plot descriptions, cast information, improved duplicate handling, and diversity-aware ranking. A larger evaluation set and structured relevance judgments would help determine whether these changes improve recommendation quality.


## Conclusion and Business Recommendations

### Conclusion

This project developed a content-based recommendation system for Netflix titles using TF-IDF vectorization and cosine similarity. The system represents titles through selected metadata, including content type, genres, director, and country, then ranks potential recommendations based on their feature similarity.

Testing across six titles showed that the recommender could consistently identify titles sharing relevant content categories and maintain separation between movies and TV shows. The evaluation also revealed limitations, including identical similarity scores, repeated language versions, and recommendations based on broad genre overlap rather than deeper thematic similarities.

Overall, the project demonstrates how machine learning techniques can support content discovery while highlighting the importance of feature selection, recommendation diversity, and systematic evaluation.

### Business Recommendations

1. **Improve content representation:** Incorporate plot descriptions, cast information, and other reliable metadata to capture similarities beyond genre labels.
2. **Promote recommendation diversity:** Reduce repetitive results and consider ranking strategies that balance relevance with variety.
3. **Evaluate recommendation quality:** Test a larger, more representative sample of titles and assess relevance and diversity using consistent evaluation criteria.
4. **Support content discovery:** Present recommendations with their shared genres and other relevant metadata to help users understand why each title was suggested.

These improvements could make the system more useful for content discovery and provide a stronger foundation for a more sophisticated recommendation platform.
